# Lab 1.6: Multi-Instance and Multi-Pass Review

You're reviewing a pull request with multiple files. Claude generated the code, and now you need Claude to review it. Sounds simple -- but *how* you set up the review determines whether it catches real bugs or rubber-stamps its own work.

**What you'll learn:**
- Why same-session self-review misses bugs (reasoning context bias)
- How independent-instance review catches what self-review misses
- Why single-pass review of large PRs causes attention dilution
- How multi-pass (local + integration) finds cross-file issues
- Statistical evidence: self-review vs independent review bug detection rates

## Setup

In [ ]:
import anthropic
import json

client = anthropic.Anthropic()  # uses ANTHROPIC_API_KEY env var
MODEL = "claude-sonnet-4-20250514"

---
## Phase 1: THE WRONG WAY -- Same-Session Self-Review

The most natural approach: ask Claude to generate code, then ask it to review its own code in the same conversation. This fails because Claude retains the **reasoning context** from generation -- it knows *why* it made each decision, making it less likely to question them.

In [ ]:
# The code we want generated has intentional bugs planted in it.
# We'll ask Claude to generate an auth module, and the generation
# will contain subtle issues.

GENERATION_PROMPT = """Write a Python user authentication module with the following functions:

1. `hash_password(password)` - Hash a password using SHA-256 with a salt
2. `verify_password(password, stored_hash)` - Verify a password against stored hash
3. `create_session(user_id)` - Create a session token with 24-hour expiry
4. `validate_session(token)` - Check if a session is valid and not expired
5. `rate_limit_login(user_id)` - Track failed attempts, lock after 5 failures

Use standard library only (hashlib, secrets, time, etc). Include proper error handling."""

print("Step 1: Generating authentication module...")
print("(This will contain subtle bugs that a good review should catch)")

In [ ]:
# Step 1: Generate the code
gen_messages = [{"role": "user", "content": GENERATION_PROMPT}]

gen_response = client.messages.create(
    model=MODEL,
    max_tokens=4000,
    messages=gen_messages
)

generated_code = gen_response.content[0].text
print("Generated code:")
print("=" * 60)
print(generated_code[:2000])
if len(generated_code) > 2000:
    print(f"\n... ({len(generated_code)} total characters)")

In [ ]:
# Step 2: WRONG -- Self-review in the same conversation
# The model has its full reasoning context from generation.

self_review_messages = gen_messages.copy()
self_review_messages.append({"role": "assistant", "content": gen_response.content})
self_review_messages.append({
    "role": "user",
    "content": (
        "Now review the code you just wrote for security bugs, logic errors, "
        "and best practice violations. Be thorough and critical. "
        "List each issue as a JSON array with objects containing: "
        "'location', 'issue', 'severity' (high/medium/low).\n\n"
        "Return ONLY the JSON array."
    )
})

print("Step 2: Running SELF-REVIEW (same session, has reasoning context)...")

self_review_response = client.messages.create(
    model=MODEL,
    max_tokens=4000,
    messages=self_review_messages
)

self_review_raw = self_review_response.content[0].text.strip()
if self_review_raw.startswith("```"):
    self_review_raw = self_review_raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

try:
    self_review_findings = json.loads(self_review_raw)
except json.JSONDecodeError:
    print(f"Parse error. Raw response:\n{self_review_raw[:500]}")
    self_review_findings = []

print(f"\nSelf-review found {len(self_review_findings)} issues:")
for i, f in enumerate(self_review_findings, 1):
    print(f"  {i}. [{f.get('severity', '?').upper()}] {f.get('location', '?')}: {f.get('issue', '')}")

---
## Phase 2: THE RIGHT WAY -- Independent Instance Review

A new Claude instance sees only the code, with no access to the reasoning that produced it. This makes it evaluate the code **on its merits** rather than through the lens of its own decisions.

In [ ]:
# Independent instance -- NEW conversation with ONLY the code

independent_review_messages = [{
    "role": "user",
    "content": (
        "Review the following Python authentication module for security bugs, "
        "logic errors, and best practice violations. Be thorough and critical.\n\n"
        "Focus on:\n"
        "- Password hashing weaknesses\n"
        "- Session management flaws\n"
        "- Race conditions\n"
        "- Input validation gaps\n"
        "- Timing attack vulnerabilities\n\n"
        f"```python\n{generated_code}\n```\n\n"
        "List each issue as a JSON array with objects containing: "
        "'location', 'issue', 'severity' (high/medium/low).\n\n"
        "Return ONLY the JSON array."
    )
}]

print("Running INDEPENDENT REVIEW (new instance, code only, no reasoning context)...")

independent_response = client.messages.create(
    model=MODEL,
    max_tokens=4000,
    messages=independent_review_messages
)

independent_raw = independent_response.content[0].text.strip()
if independent_raw.startswith("```"):
    independent_raw = independent_raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

try:
    independent_findings = json.loads(independent_raw)
except json.JSONDecodeError:
    print(f"Parse error. Raw response:\n{independent_raw[:500]}")
    independent_findings = []

print(f"\nIndependent review found {len(independent_findings)} issues:")
for i, f in enumerate(independent_findings, 1):
    print(f"  {i}. [{f.get('severity', '?').upper()}] {f.get('location', '?')}: {f.get('issue', '')}")

In [ ]:
# Compare the two approaches

self_issues = set(f.get('location', '') + ": " + f.get('issue', '')[:50] for f in self_review_findings)
indep_issues = set(f.get('location', '') + ": " + f.get('issue', '')[:50] for f in independent_findings)

only_independent = indep_issues - self_issues
only_self = self_issues - indep_issues

print("=" * 60)
print("COMPARISON: Self-Review vs Independent Review")
print("=" * 60)
print(f"\n  Self-review found:        {len(self_review_findings)} issues")
print(f"  Independent review found: {len(independent_findings)} issues")

self_high = sum(1 for f in self_review_findings if f.get('severity') == 'high')
indep_high = sum(1 for f in independent_findings if f.get('severity') == 'high')
print(f"\n  High severity (self):        {self_high}")
print(f"  High severity (independent): {indep_high}")

if len(independent_findings) > len(self_review_findings):
    diff = len(independent_findings) - len(self_review_findings)
    print(f"\n  The independent review found {diff} more issue(s).")
    print(f"  This is the reasoning context bias in action: the self-reviewer")
    print(f"  is anchored to its own reasoning and less likely to question it.")
elif len(independent_findings) == len(self_review_findings):
    print(f"\n  Same count, but check the QUALITY of findings above.")
    print(f"  Independent reviews typically find higher-severity issues.")

### Why does this happen?

When Claude generates code, it makes hundreds of micro-decisions: which hash algorithm, how to structure the session, where to add validation. In a self-review, it has access to the **reasoning behind each decision**, which acts as an anchor. It's unlikely to question decisions it just justified to itself.

An independent instance has no such anchoring. It evaluates the code purely on what it sees, making it more likely to catch:
- Assumptions that were never validated
- Edge cases that were considered and dismissed
- Patterns that seemed reasonable during generation but have subtle flaws

---
## Multi-Pass Review for Large PRs

Now let's tackle a multi-file pull request. When you send all files to a single review pass, the model allocates attention unevenly -- this is **attention dilution**. Multi-pass review solves this.

In [ ]:
# A 4-file web application with intentional issues
# Some bugs are per-file, some only visible across files

PR_FILES = {
    "auth.py": '''import hashlib
import secrets
import time

# Session store: {token: {"user_id": ..., "expires": ...}}
sessions = {}

def login(username, password):
    """Authenticate user and return session token."""
    user = db_lookup(username)  # imported from database.py
    if user and user["password_hash"] == hashlib.sha256(password.encode()).hexdigest():
        token = secrets.token_hex(32)
        sessions[token] = {
            "user_id": user["id"],
            "role": user["role"],
            "expires": time.time() + 86400  # 24 hours
        }
        return {"token": token, "role": user["role"]}
    return None

def get_session(token):
    """Return session data if token is valid and not expired."""
    session = sessions.get(token)
    if session and session["expires"] > time.time():
        return session
    return None

def logout(token):
    """Invalidate a session."""
    sessions.pop(token, None)
''',

    "routes.py": '''from auth import login, get_session
from database import get_user_data, update_user, delete_user

def handle_login(request):
    result = login(request["username"], request["password"])
    if result:
        return {"status": 200, "body": result}
    return {"status": 401, "body": "Invalid credentials"}

def handle_get_profile(request):
    session = get_session(request.get("token"))
    if not session:
        return {"status": 401, "body": "Not authenticated"}
    user_data = get_user_data(session["user_id"])
    return {"status": 200, "body": user_data}

def handle_update_profile(request):
    session = get_session(request.get("token"))
    if not session:
        return {"status": 401, "body": "Not authenticated"}
    # BUG: No authorization check — any authenticated user can update any profile
    update_user(request["user_id"], request["data"])
    return {"status": 200, "body": "Updated"}

def handle_delete_user(request):
    session = get_session(request.get("token"))
    if not session:
        return {"status": 401, "body": "Not authenticated"}
    if session["role"] == "admin":
        delete_user(request["user_id"])
        return {"status": 200, "body": "Deleted"}
    return {"status": 403, "body": "Not authorized"}
''',

    "database.py": '''import sqlite3

DB_PATH = "app.db"

def get_connection():
    return sqlite3.connect(DB_PATH)

def db_lookup(username):
    conn = get_connection()
    # BUG: SQL injection — username not parameterized
    cursor = conn.execute(f"SELECT * FROM users WHERE username = \'{username}\'")
    row = cursor.fetchone()
    conn.close()
    if row:
        return {"id": row[0], "username": row[1], "password_hash": row[2], "role": row[3]}
    return None

def get_user_data(user_id):
    conn = get_connection()
    cursor = conn.execute("SELECT id, username, email, role FROM users WHERE id = ?", (user_id,))
    row = cursor.fetchone()
    conn.close()
    if row:
        return {"id": row[0], "username": row[1], "email": row[2], "role": row[3]}
    return None

def update_user(user_id, data):
    conn = get_connection()
    # BUG: Allows updating ANY field including role (privilege escalation)
    for key, value in data.items():
        conn.execute(f"UPDATE users SET {key} = ? WHERE id = ?", (value, user_id))
    conn.commit()
    conn.close()

def delete_user(user_id):
    conn = get_connection()
    conn.execute("DELETE FROM users WHERE id = ?", (user_id,))
    conn.commit()
    conn.close()
''',

    "middleware.py": '''import time
import logging

# Rate limiting: {ip: [timestamps]}
request_log = {}
RATE_LIMIT = 100  # requests per minute

def rate_limit(request):
    """Check if request exceeds rate limit."""
    ip = request.get("ip", "unknown")
    now = time.time()
    
    if ip not in request_log:
        request_log[ip] = []
    
    # Clean old entries
    request_log[ip] = [t for t in request_log[ip] if now - t < 60]
    
    if len(request_log[ip]) >= RATE_LIMIT:
        return False
    
    request_log[ip].append(now)
    return True

def log_request(request, response):
    """Log request details."""
    # BUG: Logs sensitive data (tokens, passwords in request body)
    logging.info(f"Request: {request} -> Response: {response['status']}")

def sanitize_input(data):
    """Basic input sanitization."""
    if isinstance(data, str):
        return data.strip()
    return data
'''
}

print("=== PR FILES ===")
for name, code in PR_FILES.items():
    lines = code.strip().split("\n")
    print(f"  {name}: {len(lines)} lines")

print(f"\nPlanted bugs (you'll discover these through review):")
print(f"  Per-file bugs: SQL injection, privilege escalation, sensitive data logging")
print(f"  Cross-file bugs: auth bypass via missing authorization check + DB field update,")
print(f"    plain SHA-256 without salt, session data not cleaned on delete")

In [ ]:
# === SINGLE-PASS REVIEW (for comparison) ===

all_code = "\n".join(f"# === {name} ===\n{code}" for name, code in PR_FILES.items())

single_pass_messages = [{
    "role": "user",
    "content": (
        "Review this 4-file pull request for security bugs, logic errors, "
        "and cross-file issues. Be thorough.\n\n"
        f"```python\n{all_code}\n```\n\n"
        "List each issue as a JSON array with objects containing: "
        "'file', 'location', 'issue', 'severity' (high/medium/low), "
        "'type' (either 'local' for single-file issues or 'cross-file' for "
        "issues involving multiple files).\n\n"
        "Return ONLY the JSON array."
    )
}]

print("Running SINGLE-PASS review (all files at once)...")

single_response = client.messages.create(
    model=MODEL,
    max_tokens=4000,
    messages=single_pass_messages
)

single_raw = single_response.content[0].text.strip()
if single_raw.startswith("```"):
    single_raw = single_raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

try:
    single_findings = json.loads(single_raw)
except json.JSONDecodeError:
    print(f"Parse error. Raw response:\n{single_raw[:500]}")
    single_findings = []

print(f"\nSingle-pass found {len(single_findings)} issues:")
for i, f in enumerate(single_findings, 1):
    print(f"  {i}. [{f.get('severity','?').upper()}] {f.get('file','?')}/{f.get('location','?')}: {f.get('issue','')}")
    print(f"     Type: {f.get('type', '?')}")

In [ ]:
# === MULTI-PASS REVIEW ===

# --- Pass 1: Local review (one per file) ---
print("=== PASS 1: Local Reviews (per-file) ===")
print()

local_findings = {}

for filename, code in PR_FILES.items():
    local_messages = [{
        "role": "user",
        "content": (
            f"Review this single Python file ({filename}) for security bugs, "
            f"logic errors, and best practice violations.\n\n"
            f"```python\n{code}\n```\n\n"
            f"List each issue as a JSON array with objects containing: "
            f"'location' (function or line), 'issue', 'severity' (high/medium/low).\n\n"
            f"Return ONLY the JSON array. If no issues, return []."
        )
    }]
    
    local_response = client.messages.create(
        model=MODEL,
        max_tokens=2000,
        messages=local_messages
    )
    
    raw = local_response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    
    try:
        findings = json.loads(raw)
    except json.JSONDecodeError:
        findings = []
    
    local_findings[filename] = findings
    print(f"  {filename}: {len(findings)} issues found")
    for f in findings:
        print(f"    [{f.get('severity','?').upper()}] {f.get('location','?')}: {f.get('issue','')}")
    print()

In [ ]:
# --- Pass 2: Integration review (cross-file) ---
print("=== PASS 2: Integration Review (cross-file) ===")
print()

# Build context: all files + local findings
local_summary = ""
for filename, findings in local_findings.items():
    local_summary += f"\n{filename} local findings:\n"
    for f in findings:
        local_summary += f"  - [{f.get('severity','?')}] {f.get('location','?')}: {f.get('issue','')}\n"

integration_messages = [{
    "role": "user",
    "content": (
        "You are reviewing a 4-file Python web application. Individual file reviews "
        "have already been completed. Your job is to find CROSS-FILE issues that "
        "individual reviews would miss.\n\n"
        "Focus on:\n"
        "- Data flow between files (does auth.py's output get used safely in routes.py?)\n"
        "- API contract mismatches between files\n"
        "- Authorization gaps in the request chain\n"
        "- State management across modules\n"
        "- How bugs in one file enable exploits via another\n\n"
        "DO NOT repeat issues already found in local reviews.\n\n"
        f"=== FILES ===\n```python\n{all_code}\n```\n\n"
        f"=== LOCAL REVIEW FINDINGS (already reported, do NOT duplicate) ==="
        f"{local_summary}\n\n"
        "List ONLY new cross-file issues as a JSON array with objects containing: "
        "'files_involved' (list of filenames), 'issue', 'severity' (high/medium/low), "
        "'attack_scenario' (how an attacker would exploit this across files).\n\n"
        "Return ONLY the JSON array. If no new cross-file issues, return []."
    )
}]

integration_response = client.messages.create(
    model=MODEL,
    max_tokens=4000,
    messages=integration_messages
)

integration_raw = integration_response.content[0].text.strip()
if integration_raw.startswith("```"):
    integration_raw = integration_raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()

try:
    integration_findings = json.loads(integration_raw)
except json.JSONDecodeError:
    print(f"Parse error. Raw response:\n{integration_raw[:500]}")
    integration_findings = []

print(f"Integration pass found {len(integration_findings)} cross-file issues:\n")
for i, f in enumerate(integration_findings, 1):
    files = f.get('files_involved', [])
    print(f"  {i}. [{f.get('severity','?').upper()}] {' + '.join(files)}")
    print(f"     Issue: {f.get('issue', '')}")
    print(f"     Attack: {f.get('attack_scenario', '')}")
    print()

In [ ]:
# === COMPARE: Single-pass vs Multi-pass ===

# Count findings by type
single_local = sum(1 for f in single_findings if f.get('type') == 'local')
single_cross = sum(1 for f in single_findings if f.get('type') == 'cross-file')

multi_local_total = sum(len(f) for f in local_findings.values())
multi_cross_total = len(integration_findings)

print("=" * 60)
print("COMPARISON: Single-Pass vs Multi-Pass")
print("=" * 60)
print(f"\n{'Metric':<35} {'Single-Pass':>12} {'Multi-Pass':>12}")
print(f"{'-'*35} {'-'*12} {'-'*12}")
print(f"{'Total findings':<35} {len(single_findings):>12} {multi_local_total + multi_cross_total:>12}")
print(f"{'Local (per-file) issues':<35} {single_local:>12} {multi_local_total:>12}")
print(f"{'Cross-file issues':<35} {single_cross:>12} {multi_cross_total:>12}")

# Per-file coverage
print(f"\nPer-file coverage (multi-pass):")
for filename, findings in local_findings.items():
    single_for_file = sum(1 for f in single_findings if f.get('file') == filename)
    print(f"  {filename}: {len(findings)} (multi) vs {single_for_file} (single)")

print(f"\nMulti-pass ensures EVERY file gets thorough individual attention.")
print(f"Single-pass often focuses on the most obvious file and skims the rest.")
print(f"This is attention dilution -- a bigger context window does NOT fix it.")

---
## Phase 3: YOUR TURN -- Build a Multi-Pass Review Pipeline

Here are 4 new Python files forming a small task management API. Build the full multi-pass review pipeline: local review prompts, integration review prompt, and merged/deduplicated findings.

In [ ]:
STUDENT_PR_FILES = {
    "models.py": '''from dataclasses import dataclass
from typing import Optional
import time

@dataclass
class Task:
    id: int
    title: str
    description: str
    assignee_id: int
    status: str = "open"  # open, in_progress, done
    priority: int = 3     # 1 (highest) to 5 (lowest)
    created_at: float = 0
    
    def __post_init__(self):
        if not self.created_at:
            self.created_at = time.time()

@dataclass
class User:
    id: int
    name: str
    email: str
    role: str = "member"  # member, lead, admin
    max_tasks: int = 10
''',

    "task_service.py": '''from models import Task

# In-memory store
tasks = {}
next_id = 1

def create_task(title, description, assignee_id, priority=3):
    global next_id
    task = Task(id=next_id, title=title, description=description,
                assignee_id=assignee_id, priority=priority)
    tasks[next_id] = task
    next_id += 1
    return task

def get_task(task_id):
    return tasks.get(task_id)

def update_task_status(task_id, new_status):
    task = tasks.get(task_id)
    if task:
        # BUG: No validation of status transitions
        # (can go from "done" back to "open")
        task.status = new_status
        return task
    return None

def reassign_task(task_id, new_assignee_id):
    task = tasks.get(task_id)
    if task:
        # BUG: Doesn\'t check if new assignee exists or has capacity
        task.assignee_id = new_assignee_id
        return task
    return None

def get_user_tasks(user_id):
    return [t for t in tasks.values() if t.assignee_id == user_id]

def delete_task(task_id):
    return tasks.pop(task_id, None)
''',

    "api_handlers.py": '''from task_service import create_task, get_task, update_task_status, reassign_task, delete_task

def handle_create_task(request, session):
    if not session:
        return {"status": 401, "body": "Not authenticated"}
    task = create_task(
        title=request["title"],
        description=request["description"],
        assignee_id=request.get("assignee_id", session["user_id"]),
        priority=request.get("priority", 3)
    )
    return {"status": 201, "body": {"task_id": task.id}}

def handle_update_status(request, session):
    if not session:
        return {"status": 401, "body": "Not authenticated"}
    # BUG: Any authenticated user can update any task\'s status
    task = update_task_status(request["task_id"], request["status"])
    if task:
        return {"status": 200, "body": "Updated"}
    return {"status": 404, "body": "Task not found"}

def handle_reassign(request, session):
    if not session:
        return {"status": 401, "body": "Not authenticated"}
    # Only leads and admins can reassign
    if session.get("role") not in ("lead", "admin"):
        return {"status": 403, "body": "Not authorized"}
    task = reassign_task(request["task_id"], request["new_assignee_id"])
    if task:
        return {"status": 200, "body": "Reassigned"}
    return {"status": 404, "body": "Task not found"}

def handle_delete_task(request, session):
    if not session:
        return {"status": 401, "body": "Not authenticated"}
    # Only admins can delete
    if session.get("role") != "admin":
        return {"status": 403, "body": "Not authorized"}
    task = delete_task(request["task_id"])
    if task:
        return {"status": 200, "body": "Deleted"}
    return {"status": 404, "body": "Task not found"}
''',

    "notifications.py": '''import smtplib

SMTP_HOST = "smtp.company.com"
SMTP_PORT = 587

def notify_assignment(task, assignee_email):
    """Send email when task is assigned."""
    subject = f"New task: {task.title}"
    body = f"You have been assigned task #{task.id}: {task.title}\\n{task.description}"
    _send_email(assignee_email, subject, body)

def notify_status_change(task, old_status):
    """Notify relevant parties of status change."""
    # BUG: No way to look up assignee email from assignee_id
    # This function can\'t actually send the notification
    subject = f"Task #{task.id} status: {old_status} -> {task.status}"
    body = f"Task \'{task.title}\' status changed."
    # _send_email(???, subject, body)  # Can\'t resolve email!

def _send_email(to_addr, subject, body):
    """Send email via SMTP."""
    # BUG: No error handling for SMTP failures
    server = smtplib.SMTP(SMTP_HOST, SMTP_PORT)
    server.starttls()
    server.sendmail("tasks@company.com", to_addr, f"Subject: {subject}\\n\\n{body}")
    server.quit()
'''
}

print("=== STUDENT PR FILES ===")
for name, code in STUDENT_PR_FILES.items():
    lines = code.strip().split("\n")
    print(f"  {name}: {len(lines)} lines")

print("\nYour tasks:")
print("  1. Write local review prompts (one per file)")
print("  2. Run them and collect local findings")
print("  3. Write an integration review prompt that takes all files + local findings")
print("  4. Merge and deduplicate all findings")

In [ ]:
# ============================================================
# TASK 1: Run local reviews for each file
# ============================================================
# Write a function that takes a filename and code, sends a
# review prompt, and returns parsed findings.
#
# Hint: Use a prompt structure similar to Phase 2's local pass.
# Make sure to ask for JSON output.

def review_file(filename, code):
    """Review a single file and return findings as a list of dicts."""
    # YOUR CODE HERE
    # 1. Build a review prompt for this specific file
    # 2. Call client.messages.create()
    # 3. Parse the JSON response
    # 4. Return the list of findings
    pass  # Replace with your implementation


# Run local reviews
# student_local_findings = {}
# for filename, code in STUDENT_PR_FILES.items():
#     findings = review_file(filename, code)
#     student_local_findings[filename] = findings or []
#     print(f"{filename}: {len(student_local_findings[filename])} issues")
#     for f in student_local_findings[filename]:
#         print(f"  [{f.get('severity','?')}] {f.get('location','?')}: {f.get('issue','')}")
#     print()

In [ ]:
# ============================================================
# TASK 2: Run integration review
# ============================================================
# Write a function that takes all files + local findings and
# finds cross-file issues.
#
# Key: Tell the model NOT to duplicate local findings.
# Ask it to focus on data flow, API contracts, and cross-file
# authorization gaps.

def integration_review(files, local_findings):
    """Review cross-file interactions and return new findings."""
    # YOUR CODE HERE
    # 1. Combine all files into one code block
    # 2. Summarize local findings so the model knows what's already found
    # 3. Ask for ONLY cross-file issues
    # 4. Return parsed findings
    pass  # Replace with your implementation


# Run integration review
# student_cross_findings = integration_review(STUDENT_PR_FILES, student_local_findings)
# print(f"Cross-file issues: {len(student_cross_findings or [])}")
# for f in (student_cross_findings or []):
#     print(f"  [{f.get('severity','?')}] {f.get('files_involved','?')}: {f.get('issue','')}")

In [ ]:
# ============================================================
# TASK 3: Merge and deduplicate
# ============================================================

def merge_findings(local_findings_by_file, cross_findings):
    """Merge local and cross-file findings, removing duplicates."""
    # YOUR CODE HERE
    # 1. Collect all local findings, tagged with their source file
    # 2. Add cross-file findings, tagged as "cross-file"
    # 3. Deduplicate (if any finding appears in both local and cross,
    #    keep only the cross-file version as it has more context)
    # 4. Return the merged list
    pass  # Replace with your implementation


# Merge
# all_findings = merge_findings(student_local_findings, student_cross_findings)
# print(f"Total unique findings: {len(all_findings or [])}")

In [ ]:
# === CHECKER ===
# Run this after completing Tasks 1-3 above

EXPECTED_LOCAL_BUGS = {
    "task_service.py": ["no status transition validation", "no assignee capacity check"],
    "api_handlers.py": ["any user can update any task status"],
    "notifications.py": ["can't resolve assignee email", "no SMTP error handling"],
}

EXPECTED_CROSS_FILE_BUGS = [
    "reassign doesn't check User.max_tasks (api_handlers + task_service + models)",
    "status change doesn't trigger notification (api_handlers + notifications)",
    "notifications.py has no way to look up user email from assignee_id (needs database access)",
]

print("=== EXPECTED FINDINGS ===")
print("\nLocal bugs your reviews should find:")
for f, bugs in EXPECTED_LOCAL_BUGS.items():
    for b in bugs:
        print(f"  {f}: {b}")

print("\nCross-file bugs the integration pass should find:")
for b in EXPECTED_CROSS_FILE_BUGS:
    print(f"  {b}")

print("\nCheck your findings against these lists.")
print("A good multi-pass review should catch most of the local bugs")
print("AND find cross-file issues that single-pass reviews miss.")

---
## Phase 4: STRESS TEST -- Statistical Comparison

Let's run self-review vs independent review multiple times and count bugs found. Then compare single-pass vs multi-pass coverage.

In [ ]:
# === STRESS TEST 1: Self-review vs Independent review (3 runs each) ===

# Use the auth code from Phase 1
REVIEW_TARGET = generated_code

print("=== STRESS TEST: Self-Review vs Independent Review ===")
print("Running 3 trials of each...\n")

self_review_counts = []
independent_counts = []

for trial in range(3):
    print(f"--- Trial {trial + 1} ---")
    
    # Self-review: generate then review in same session
    gen_msgs = [{"role": "user", "content": GENERATION_PROMPT}]
    gen_resp = client.messages.create(model=MODEL, max_tokens=4000, messages=gen_msgs)
    code = gen_resp.content[0].text
    
    self_msgs = gen_msgs.copy()
    self_msgs.append({"role": "assistant", "content": gen_resp.content})
    self_msgs.append({
        "role": "user",
        "content": (
            "Review the code you just wrote for security bugs, logic errors, "
            "and best practice violations. Return a JSON array with objects "
            "containing 'location', 'issue', 'severity'. Return ONLY the JSON array."
        )
    })
    
    self_resp = client.messages.create(model=MODEL, max_tokens=4000, messages=self_msgs)
    self_raw = self_resp.content[0].text.strip()
    if self_raw.startswith("```"):
        self_raw = self_raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    try:
        self_findings = json.loads(self_raw)
    except json.JSONDecodeError:
        self_findings = []
    
    self_high = sum(1 for f in self_findings if f.get('severity') == 'high')
    self_review_counts.append({"total": len(self_findings), "high": self_high})
    
    # Independent review: new session, only the code
    indep_msgs = [{
        "role": "user",
        "content": (
            f"Review this Python authentication module for security bugs, logic errors, "
            f"and best practice violations. Be thorough.\n\n"
            f"```python\n{code}\n```\n\n"
            f"Return a JSON array with objects containing 'location', 'issue', 'severity'. "
            f"Return ONLY the JSON array."
        )
    }]
    
    indep_resp = client.messages.create(model=MODEL, max_tokens=4000, messages=indep_msgs)
    indep_raw = indep_resp.content[0].text.strip()
    if indep_raw.startswith("```"):
        indep_raw = indep_raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    try:
        indep_findings = json.loads(indep_raw)
    except json.JSONDecodeError:
        indep_findings = []
    
    indep_high = sum(1 for f in indep_findings if f.get('severity') == 'high')
    independent_counts.append({"total": len(indep_findings), "high": indep_high})
    
    print(f"  Self-review:   {len(self_findings):2d} total, {self_high} high")
    print(f"  Independent:   {len(indep_findings):2d} total, {indep_high} high")

print(f"\n=== SUMMARY ACROSS 3 TRIALS ===")
avg_self = sum(r['total'] for r in self_review_counts) / 3
avg_indep = sum(r['total'] for r in independent_counts) / 3
avg_self_high = sum(r['high'] for r in self_review_counts) / 3
avg_indep_high = sum(r['high'] for r in independent_counts) / 3

print(f"{'Metric':<30} {'Self-Review':>12} {'Independent':>12}")
print(f"{'-'*30} {'-'*12} {'-'*12}")
print(f"{'Avg total findings':<30} {avg_self:>12.1f} {avg_indep:>12.1f}")
print(f"{'Avg high-severity':<30} {avg_self_high:>12.1f} {avg_indep_high:>12.1f}")

if avg_indep > avg_self:
    print(f"\nIndependent review found {avg_indep - avg_self:.1f} more issues on average.")
    print(f"The reasoning context bias is measurable and consistent.")
else:
    print(f"\nResults were close -- but check the SEVERITY distribution above.")
    print(f"Independent reviews typically find more high-severity issues.")

In [ ]:
# === STRESS TEST 2: Single-pass vs Multi-pass coverage ===

print("=== STRESS TEST: Single-Pass vs Multi-Pass on 4-File PR ===")
print()

# Run single-pass 3 times
single_pass_runs = []
for trial in range(3):
    sp_messages = [{
        "role": "user",
        "content": (
            "Review this 4-file pull request for security bugs, logic errors, "
            "and cross-file issues.\n\n"
            f"```python\n{all_code}\n```\n\n"
            "List each issue as a JSON array with objects containing: "
            "'file', 'location', 'issue', 'severity', "
            "'type' ('local' or 'cross-file'). Return ONLY the JSON array."
        )
    }]
    
    sp_resp = client.messages.create(model=MODEL, max_tokens=4000, messages=sp_messages)
    sp_raw = sp_resp.content[0].text.strip()
    if sp_raw.startswith("```"):
        sp_raw = sp_raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    try:
        sp_findings = json.loads(sp_raw)
    except json.JSONDecodeError:
        sp_findings = []
    
    files_covered = set(f.get('file', '') for f in sp_findings)
    cross = sum(1 for f in sp_findings if f.get('type') == 'cross-file')
    single_pass_runs.append({
        "total": len(sp_findings),
        "files_covered": len(files_covered),
        "cross_file": cross
    })
    print(f"  Single-pass trial {trial+1}: {len(sp_findings)} findings, "
          f"{len(files_covered)} files covered, {cross} cross-file")

print()

# Multi-pass stats (use the findings from Phase 2)
mp_total = multi_local_total + multi_cross_total
mp_files = len(local_findings)  # Always covers all files

print(f"  Multi-pass result: {mp_total} findings, "
      f"{mp_files} files covered, {multi_cross_total} cross-file")

avg_sp_total = sum(r['total'] for r in single_pass_runs) / 3
avg_sp_files = sum(r['files_covered'] for r in single_pass_runs) / 3
avg_sp_cross = sum(r['cross_file'] for r in single_pass_runs) / 3

print(f"\n{'Metric':<35} {'Single-Pass (avg)':>18} {'Multi-Pass':>12}")
print(f"{'-'*35} {'-'*18} {'-'*12}")
print(f"{'Total findings':<35} {avg_sp_total:>18.1f} {mp_total:>12}")
print(f"{'Files with findings':<35} {avg_sp_files:>18.1f} {mp_files:>12}")
print(f"{'Cross-file issues':<35} {avg_sp_cross:>18.1f} {multi_cross_total:>12}")

print(f"\nMulti-pass guarantees every file is reviewed individually,")
print(f"then finds cross-file issues in a dedicated integration pass.")
print(f"Single-pass suffers from attention dilution -- inconsistent")
print(f"coverage across files and often misses cross-file data flows.")
print(f"\nA bigger context window does NOT fix attention dilution.")
print(f"The solution is structured passes, not more tokens.")

---
## Key Takeaways

1. **Self-review in the same session is biased.** The model retains reasoning context from generation, making it less likely to question its own decisions. Always use an independent instance.
2. **Independent instance = new conversation with only the code.** No generation context, no reasoning anchoring. Evaluates the code on its merits.
3. **Single-pass review of large PRs causes attention dilution.** The model focuses unevenly across files, missing issues in files it gives less attention.
4. **Multi-pass solves attention dilution:** Pass 1 reviews each file individually (consistent depth), Pass 2 reviews cross-file interactions (data flow, API contracts, authorization chains).
5. **"Use a bigger context window" is wrong.** More tokens does not fix uneven attention allocation. Structured passes are the answer.
6. **On the exam:** When asked how to improve code review quality for large PRs, look for multi-pass/multi-instance answers, not bigger models or longer contexts.

## Module 1 Complete

Continue to Module 2: Context Management and Reliability